# Federated Runtime 101: Quickstart with MNIST

Welcome to the first **FederatedRuntime** Tutorial ! 
This tutorial demonstrates how to deploy a Workflow Interface based Federated-Learning experiment on a distributed computing infrastructure.

Data scientists often begin by developing and fine-tuning machine learning models in a local computing environment before deploying to a Federated setup. OpenFL supports this methodology and the Tutorial guides the user through the following steps:
- **Simulate** a Federated Learning experiment locally using **LocalRuntime** 
- **Deploy** this experiment on Federated Infrastructure using **FederatedRuntime**

References: 
The tutorial builds on following features  
1. **Workflow-Interface** and **LocalRuntime**: Simulate Federated Learning experiments using Workflow-Interface. Explore [101 MNIST](https://github.com/securefederatedai/openfl/blob/develop/openfl-tutorials/experimental/LocalRuntime/101_MNIST.ipynb) for insights
2. **Jupyter notebook annotation & export directives**: Refer to [1001 Workspace Creation from JupyterNotebook](https://github.com/securefederatedai/openfl/blob/develop/openfl-tutorials/experimental/1001_Workspace_Creation_from_JupyterNotebook.ipynb)

Let's get started !


# Getting Started

Initially, we start by specifying the module where cells marked with the `#| export` directive will be automatically exported. 

In the following cell, `#| default_exp experiment `indicates that the exported file will be named 'experiment'. This name can be modified based on user's requirement & preferences

In [ ]:
#| default_exp experiment

Once we have specified the name of the module, subsequent cells of the notebook need to be *appended* by the `#| export` directive as shown below. User should ensure that *all* the notebook functionality required in the Federated Learning experiment is included in this directive

We start by installing OpenFL and dependencies of the workflow interface 
> These dependencies are required to be exported and become the requirements for the Federated Learning Workspace 

In [ ]:
#| export

# !pip install git+https://github.com/ishant162/openfl.git@experimental-director-workflow
# !pip install -r ../../../workflow_interface_requirements.txt
# !pip install torch
# !pip install torchvision
# !pip install -U ipywidgets


# Model definition

We begin with the quintessential example of a small pytorch CNN model trained on the MNIST dataset. Let's start define our hyperparameters, dataloaders, model and helper functions to train and validate the model like we would for any other deep learning experiment

In [ ]:
# | export

import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torch
import torchvision
import numpy as np

# Hyperparameters
learning_rate = 0.01
log_interval = 10

mnist_train = torchvision.datasets.MNIST(
    "../files/",
    train=True,
    download=True,
    transform=torchvision.transforms.Compose(
        [
            torchvision.transforms.ToTensor(),
            torchvision.transforms.Normalize((0.1307,), (0.3081,)),
        ]
    ),
)

mnist_test = torchvision.datasets.MNIST(
    "../files/",
    train=False,
    download=True,
    transform=torchvision.transforms.Compose(
        [
            torchvision.transforms.ToTensor(),
            torchvision.transforms.Normalize((0.1307,), (0.3081,)),
        ]
    ),
)


class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        self.conv1 = nn.Conv2d(1, 10, kernel_size=5)
        self.conv2 = nn.Conv2d(10, 20, kernel_size=5)
        self.conv2_drop = nn.Dropout2d()
        self.fc1 = nn.Linear(320, 50)
        self.fc2 = nn.Linear(50, 10)

    def forward(self, x):
        x = F.relu(F.max_pool2d(self.conv1(x), 2))
        x = F.relu(F.max_pool2d(self.conv2_drop(self.conv2(x)), 2))
        x = x.view(-1, 320)
        x = F.relu(self.fc1(x))
        x = F.dropout(x, training=self.training)
        x = self.fc2(x)
        return F.log_softmax(x)


# def inference(network, test_loader):
#     network.eval()
#     test_loss = 0
#     correct = 0
#     with torch.no_grad():
#         for data, target in test_loader:
#             output = network(data)
#             test_loss += F.nll_loss(output, target, size_average=False).item()
#             pred = output.data.max(1, keepdim=True)[1]
#             correct += pred.eq(target.data.view_as(pred)).sum()
#     test_loss /= len(test_loader.dataset)
#     print(
#         "\nTest set: Avg. loss: {:.4f}, Accuracy: {}/{} ({:.0f}%)\n".format(
#             test_loss, correct, len(test_loader.dataset), 100.0 * correct / len(test_loader.dataset)
#         )
#     )
#     accuracy = float(correct / len(test_loader.dataset))
#     return accuracy


def validate(model, test_loader):
    # Helper function to validate the model
    model.eval()
    correct = 0
    with torch.no_grad():
        for data, target in test_loader:
            output = model(data)
            pred = output.data.max(1, keepdim=True)[1]
            correct += pred.eq(target.data.view_as(pred)).sum()
    accuracy = float(correct / len(test_loader.dataset))
    return accuracy


def train_model(model, optimizer, data_loader, caller, round_number, log=False):
    # Helper function to train the model
    train_loss = 0
    model.train()
    for batch_idx, (X, y) in enumerate(data_loader):
        optimizer.zero_grad()

        output = model(X)
        loss = F.nll_loss(output, y)
        loss.backward()

        optimizer.step()

        train_loss += loss.item() * len(X)
        if batch_idx % log_interval == 0 and log:
            print(
                "{:<20} Train Epoch: {:<3} [{:<3}/{:<4} ({:<.0f}%)] Loss: {:<.6f}".format(
                    entity,
                    round_number,
                    batch_idx * len(X),
                    len(data_loader.dataset),
                    100.0 * batch_idx / len(data_loader),
                    loss.item(),
                )
            )
    train_loss /= len(data_loader.dataset)
    return train_loss

# Workflow definition

Next we import the `FLSpec`, placement decorators (`aggregator/collaborator`), and define the `FedAvg` helper function

- `FLSpec` – Defines the flow specification. User defined flows are subclasses of this.
- `aggregator/collaborator` - placement decorators that define where the task will be assigned
- `FedAvg` - helper function for Federated Averaging


In [ ]:
#| export

from copy import deepcopy

from openfl.experimental.interface import FLSpec
from openfl.experimental.placement import aggregator, collaborator

def FedAvg(agg_model, models, weights=None):
    # Helper function for federated averaging
    state_dicts = [model.state_dict() for model in models]
    state_dict = agg_model.state_dict()
    for key in models[0].state_dict():
        state_dict[key] = torch.from_numpy(np.average([state[key].numpy() for state in state_dicts],
                                                      axis=0, 
                                                      weights=weights))
        
    agg_model.load_state_dict(state_dict)
    return agg_model

Let us now define the Workflow. Here we use the same tasks as the [quickstart](https://github.com/securefederatedai/openfl/blob/develop/openfl-tutorials/experimental/workflow/LocalRuntime/101_MNIST.ipynb)

In [ ]:
# | export

class FederatedFlow_TorchMNIST(FLSpec):
    """
    This Flow trains a CNN on MNIST Model in Federated Learning
    """

    def __init__(self, model=None, optimizer=None, learning_rate=1e-2, rounds=3, **kwargs):
        super().__init__(**kwargs)

        self.model = model
        self.optimizer = optimizer
        self.learning_rate = learning_rate
        self.rounds = rounds
        self.results = []

    @aggregator
    def start(self):
        """
        This is the start of the Flow.
        """
        print(f"Initializing Workflow .... ")

        self.collaborators = self.runtime.collaborators
        self.current_round = 0

        self.next(self.aggregated_model_validation, foreach="collaborators")

    @collaborator
    def aggregated_model_validation(self):
        """
        Perform validation of aggregated model on collaborators.
        """
        print(f"<Collab: {self.input}> Performing aggregated model validation")
        self.agg_validation_score = validate(self.model, self.test_loader)
        print(
            f"<Collab: {self.input}> Aggregated Model validation score = {self.agg_validation_score}"
        )

        self.next(self.train)

    @collaborator
    def train(self):
        """
        Train model on Local collaborator dataset.
        """
        print("<Collab>: Training Model on local dataset ... ")

        self.optimizer = optim.SGD(self.model.parameters(), lr=self.learning_rate)#, momentum=self.momentum)
        train_loss = 0
        self.model.train()

        for batch_idx, (data, target) in enumerate(self.train_loader):

            self.optimizer.zero_grad()

            output = self.model(data)
            loss = F.nll_loss(output, target)
            loss.backward()

            self.optimizer.step()

            train_loss = loss.item() * len(data)
            if batch_idx % log_interval == 0:
                print(
                    "Train Epoch: 1 [{:6}/{:6} ({:.0f}%)]\tLoss: {:.6f}".format(
                        batch_idx * len(data),
                        len(self.train_loader.dataset),
                        100.0 * batch_idx / len(self.train_loader),
                        loss.item(),
                    )
                )

        self.loss = train_loss / len(self.train_loader.dataset)

        self.training_completed = True
        self.next(self.local_model_validation)

    @collaborator
    def local_model_validation(self):
        """
        Validate locally trained model.
        """
        self.local_validation_score = validate(self.model, self.test_loader)
        print(
            f"<Collab: {self.input}> Local model validation score = {self.local_validation_score}"
        )
        self.next(self.join)

    @aggregator
    def join(self, inputs):
        """
        Model aggregation step.
        """
        print(f"<Agg>: Joining models from collaborators...")

        # Average Training loss, aggregated and locally trained model accuracy 
        self.average_loss = sum(input.loss for input in inputs) / len(inputs)
        self.aggregated_model_accuracy = sum(input.agg_validation_score for input in inputs) / len(inputs)
        self.local_model_accuracy = sum(input.local_validation_score for input in inputs) / len(inputs)

        print(
            f"   Aggregated model validation score = {self.aggregated_model_accuracy}"
        )
        print(f"   Average training loss = {self.average_loss}")
        print(f"   Average local model validation values = {self.local_model_accuracy}")

        self.model = FedAvg(self.model, [input.model for input in inputs])
        self.optimizer = [input.optimizer for input in inputs][0]

        self.results.append(
            [
                self.current_round,
                self.aggregated_model_accuracy,
                self.average_loss,
                self.local_model_accuracy,
            ]
        )

        self.current_round += 1
        if self.current_round < self.rounds:
            self.next( self.aggregated_model_validation, foreach="collaborators")
        else:
            self.next(self.end)

    @aggregator
    def end(self):
        """
        This is the last step in the Flow.
        """

        print(f"This is the end of the flow")

# Simulation: LocalRuntime

We now import the `LocalRuntime`, participants (`Aggregator/Collaborator`), and initialize the private attributes for participants

- `Runtime` – Defines where the flow runs, infrastructure for task transitions (how information gets sent). The `LocalRuntime` simulates the flow on local node.
- `Aggregator/Collaborator` - Participants in the simulation


In [ ]:
#| export

import random 
from openfl.experimental.interface import Aggregator, Collaborator
from openfl.experimental.runtime import LocalRuntime

# Setup Aggregator & initialize private attributes
aggregator = Aggregator()
aggregator.private_attributes = {}

# Setup Collaborators & initialize shards of MNIST dataset as private attributes 
batch_size = 32
n_collaborators = 4
collaborator_names = ['Portland', 'Seattle']

collaborators = [Collaborator(name=name) for name in collaborator_names]
for idx, collaborator in enumerate(collaborators):
    local_train = deepcopy(mnist_train)
    local_test = deepcopy(mnist_test)
    local_train.data = mnist_train.data[idx:10000:n_collaborators]
    local_train.targets = mnist_train.targets[idx:10000:n_collaborators]
    local_test.data = mnist_test.data[idx:1000:n_collaborators]
    local_test.targets = mnist_test.targets[idx:1000:n_collaborators]
    collaborator.private_attributes = {
            'train_loader': torch.utils.data.DataLoader(local_train,batch_size=batch_size, shuffle=False),
            'test_loader': torch.utils.data.DataLoader(local_test,batch_size=batch_size, shuffle=False)
    }

local_runtime = LocalRuntime(aggregator=aggregator, collaborators=collaborators, backend='single_process')
print(f'Local runtime collaborators = {local_runtime.collaborators}')

### Start Simulation

Now that we have our flow and runtime defined, let's run the simulation ! 

In [ ]:
#| export

#TODO: For reproducibility
random_seed = 1
torch.manual_seed(random_seed)
np.random.seed(random_seed)
random.seed(random_seed)

model = Net()
optimizer = optim.SGD(model.parameters(), lr=learning_rate)
flflow = FederatedFlow_TorchMNIST(model, optimizer, learning_rate, rounds=5, checkpoint=False)
flflow.runtime = local_runtime
flflow.run()

Let us check the simulation results

In [ ]:
import pandas as pd

column_names = pd.DataFrame([["Rounds", ""],
                             ["LocalRuntime", "Agg Model Acc."], 
                             ["LocalRuntime", "Avg Train Loss"], 
                             ["LocalRuntime", "Avg Train Acc"]])

columns = pd.MultiIndex.from_frame(column_names)

df_local_runtime = pd.DataFrame(flflow.results, columns=columns)
print('*********** Simulation Results *********** ')
display(df_local_runtime)

# Deploy: FederatedRuntime

We now import `FederatedRuntime` that enables deployment of experiment on distributed infrastructure. Initializing the `FederatedRuntime` requires following inputs to be provided by the user:

- `director_info` – director information including fqdn of the director node, port, and certificate information
- `fed_collaborators` - names of the collaborators participating in experiment
- 'notebook_path' - path to this jupyter notebook


In [ ]:
#| export

from openfl.experimental.runtime import FederatedRuntime

director_info = {
    'director_node_fqdn':'localhost',
    'director_port':50050,
    'cert_chain': None,
    'api_cert': None,
    'api_private_key': None,
}

fed_collaborators = ['Portland', 'Seattle']
federated_runtime = FederatedRuntime(
    collaborators=fed_collaborators,
    director=director_info, 
    notebook_path='./101_MNIST_FederatedRuntime.ipynb'
)

## Setup Federation

Let us create the distributed infrastructure and start the Director node and Envoys as described in [README](fixme: link)

Let us check if the envoys are connected to the director by using the `get_envoys` method of `FederatedRuntime`

In [ ]:
federated_runtime.get_envoys()

Now that we have our `FederatedRuntime` and distributed infrastructure ready, let us deploy the Federated learning experiment ! 


In [ ]:
#| export

import random

random_seed = 1
torch.manual_seed(random_seed)
np.random.seed(random_seed)
random.seed(random_seed)

model = Net()
optimizer = optim.SGD(model.parameters(), lr=learning_rate)
fed_flow = FederatedFlow_TorchMNIST(model, optimizer, learning_rate, rounds=5, checkpoint=False)
fed_flow.runtime = federated_runtime
fed_flow.run()



Let us check the results of experiment from FederatedRuntime

In [ ]:
import pandas as pd

column_names = pd.DataFrame([["Rounds", ""],
                             ["FederatedRuntime", "Agg Model Acc."], 
                             ["FederatedRuntime", "Avg Train Loss"], 
                             ["FederatedRuntime", "Avg Train Acc"]])

columns = pd.MultiIndex.from_frame(column_names)

df_fed = pd.DataFrame(fed_flow.results, columns=columns)
print('*********** Federated Runtime Results *********** ')
display(df_fed)


Finally, we compare the Simulation & Federation results

In [ ]:
print('*********** Comparison of Simulation & Federation esults *********** ')
df_combined = pd.merge(df_local_runtime, df_fed, on=["Rounds"])
display(df_combined)
